In [5]:
import pandas as pd
import numpy as np

# Read the raw Excel file (merged cells for Group, repeated rows for replicates)
raw = pd.read_excel("../data/raw_biochemistry_data.xlsx", header=0)
raw.columns = ["Group", "Replicate", "MDA", "GSH", "GpX", "GRD", "SOD"]

# Forward-fill the merged "Group" cells, drop empty separator rows
raw["Group"] = raw["Group"].ffill()
df = raw.dropna(subset=["MDA"]).copy()

# Standardise group names (fix "Group4" -> "Group 4", strip spaces)
df["Group"] = df["Group"].str.replace(r"Group\s*", "Group ", regex=True).str.strip()

# Map numeric group codes to descriptive names (matching your thesis)
group_map = {
    "Group 1": "Normal Control",
    "Group 2": "Atherosclerotic Control",
    "Group 3": "Low-Dose NAC",
    "Group 4": "High-Dose NAC",
    "Group 5": "NAC Treatment",
}
df["Group"] = df["Group"].map(group_map)

# Assign a clean unique Animal_ID per row within each group
df["Animal_ID"] = df.groupby("Group").cumcount() + 1
df["Animal_ID"] = df["Group"].str[:3].str.upper() + "_" + df["Animal_ID"].astype(str)

# Reorder and reset index
df = df[["Group", "Animal_ID", "MDA", "GSH", "GpX", "GRD", "SOD"]].reset_index(drop=True)

print(f"✅ Cleaned dataset: {df.shape[0]} animals × {df.shape[1]} columns")
print(f"Groups found: {df['Group'].unique().tolist()}")
df

✅ Cleaned dataset: 25 animals × 7 columns
Groups found: ['Normal Control', 'Atherosclerotic Control', 'Low-Dose NAC', 'High-Dose NAC', 'NAC Treatment']


,Group,Animal_ID,MDA,GSH,GpX,GRD,SOD
0,Normal Control,NOR_1,168.0,42.900,121.0,62.0,11.20
1,Normal Control,NOR_2,138.0,58.000,119.0,40.0,13.40
2,Normal Control,NOR_3,162.0,45.600,108.0,48.0,9.40
3,Normal Control,NOR_4,145.0,46.600,111.6,53.0,14.10
4,Normal Control,NOR_5,144.0,57.400,121.6,49.0,12.80
5,Atherosclerotic Control,ATH_1,284.0,22.800,81.0,22.0,1.50
6,Atherosclerotic Control,ATH_2,279.0,23.100,82.8,18.0,2.30
7,Atherosclerotic Control,ATH_3,249.0,25.978,73.6,25.0,3.10
8,Atherosclerotic Control,ATH_4,275.0,27.808,66.3,26.4,1.87
9,Atherosclerotic Control,ATH_5,254.0,24.600,81.4,28.0,2.30


In [6]:
published_means = {
    "Normal Control":          {"MDA":151.40,"GSH":50.10,"GpX":116.24,"GRD":50.40,"SOD":12.18},
    "Atherosclerotic Control": {"MDA":268.20,"GSH":24.86,"GpX":77.02, "GRD":23.88,"SOD":2.21},
    "Low-Dose NAC":            {"MDA":182.00,"GSH":40.68,"GpX":92.34, "GRD":36.08,"SOD":4.78},
    "High-Dose NAC":           {"MDA":160.00,"GSH":46.43,"GpX":101.08,"GRD":46.40,"SOD":9.00},
    "NAC Treatment":           {"MDA":215.20,"GSH":33.98,"GpX":90.12, "GRD":28.70,"SOD":3.34},
}

computed_means = df.groupby("Group")[["MDA","GSH","GpX","GRD","SOD"]].mean().round(2)

print("Computed means from raw data:")
display(computed_means)

print("\nDifference vs. published thesis values (should be ~0):")
for group, vals in published_means.items():
    for marker, pub_val in vals.items():
        diff = computed_means.loc[group, marker] - pub_val
        if abs(diff) > 0.5:
            print(f"⚠️  {group} / {marker}: computed={computed_means.loc[group,marker]}, published={pub_val}, diff={diff:.2f}")

print("\n✅ If no warnings printed above, the raw data perfectly matches your thesis.")

Computed means from raw data:


,MDA,GSH,GpX,GRD,SOD
Group,,,,,
Atherosclerotic Control,268.2,24.86,77.02,23.88,2.21
High-Dose NAC,160.0,46.43,101.08,46.40,9.00
Low-Dose NAC,182.0,40.68,92.34,36.08,4.78
NAC Treatment,215.2,33.98,90.12,28.70,3.34
Normal Control,151.4,50.10,116.24,50.40,12.18



Difference vs. published thesis values (should be ~0):

✅ If no warnings printed above, the raw data perfectly matches your thesis.


In [8]:
df.to_csv("../data/oxidative_stress_data.csv", index=False)
print("✅ Saved clean dataset to: ../data/oxidative_stress_data_.csv")
df.describe()

✅ Saved clean dataset to: ../data/oxidative_stress_data_.csv


,MDA,GSH,GpX,GRD,SOD
count,25.000000,25.00000,25.00000,25.000000,25.000000
mean,195.360000,39.21056,95.36000,37.092000,6.302800
std,49.038998,10.62258,15.03942,11.163554,3.963031
min,113.000000,22.80000,66.30000,18.000000,1.500000
25%,162.000000,31.30000,84.60000,28.000000,2.800000
50%,184.000000,39.65800,96.20000,37.600000,5.200000
75%,225.000000,46.10000,103.80000,46.400000,9.400000
max,284.000000,58.00000,121.60000,62.000000,14.100000
